# Lab 07 · Agentic Retrieval (Notebook Walkthrough)

**Concept.** Agentic retrieval uses Azure AI Search's knowledge base: a planning model decomposes the question into **subqueries**, runs them across the knowledge sources, and returns grounded references that are synthesized into a cited answer. It is the strongest mode for **multi-hop** reasoning that spans sections.

We run it against the generatively-enriched index (best material for the planner).

## Step 1 — Bootstrap

In [1]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
sys.path.insert(0, str(NB_DIR if (NB_DIR / 'lab_runtime.py').exists() else NB_DIR / 'notebooks'))
import lab_runtime as lab

info = lab.bootstrap()
info

{'repo_root': 'C:\\Users\\samsonlee\\rag-on-azure-lab',
 'env_values_loaded': 89,
 'search_endpoint': 'https://aisearchlabl4qxbh.search.windows.net',
 'search_configured': True,
 'embedding_deployment': 'text-embedding-3-large-vector',
 'chat_deployment': 'gpt-5-4-mini-chat',
 'agentic_planning_model': 'gpt-5-4-mini-search',
 'canonical_index': 'ai-search-lab-index'}

## Step 2 — Use the enriched corpus

In [2]:
job = lab.ingest(skill_profile='genai_enrichment', reuse=True)
lab.chunk_overview(job)

Reusing existing 'genai_enrichment' ingestion (doc_id=0262a928, 412 chunks). Pass reuse=False to force a fresh run.


{'doc_id': '0262a928d6da473295bee04acb14c024',
 'skill_profile': 'genai_enrichment',
 'chunk_count': 412,
 'avg_tokens': 200.4,
 'max_tokens': 777,
 'chunks_with_summary': 412,
 'chunks_with_keyword_hints': 412,
 'chunks_with_image_description': 0}

## Step 3 — Ask a multi-hop question

Q2 requires combining groundwater behaviour, clay-layer effects, and design controls — evidence that lives in different sections.

In [3]:
Q2 = 'Given a site with high groundwater and clay layers, what are the key excavation risks and design considerations?'

resp = lab.ask(Q2, job=job, retrieval_mode='agentic', record_as='lab07_agentic_q2')
lab.show_answer(resp, max_citations=8)

[retrieval_mode=agentic | scoring_profile=default | citations=8]

Key excavation risks and design considerations for a site with **high groundwater** and **clay layers** include: - **Overall instability**: Excavations near a steeply sloping site with a high groundwater table are at higher risk of loss of overall stability, especially where a weak layer such as loose sand or soft clay lies below the embedded wall. [1]
- **Consolidation settlement in clay**: In low-permeability fine-grained soils such as marine clay, changes in porewater pressure can cause consolidation settlement, and the delayed effects can be important on longer projects. [2]
- **Differential pore pressures between layers**: If a low-permeability clay layer separates groundwater in an upper fill from sand below, seepage and dewatering can create differential piezometric pressures that trigger consolidation settlement in the clay, even if the upper groundwater level stays stable. [3][2]
- **Uplift / base heave**: If a 

## Step 4 — Inspect the planner activity

The diagnostics expose the subqueries the planner issued and how many references each returned.

In [4]:
diag = resp.diagnostics or {}
print('mode:', diag.get('mode'))
print('answer synthesis enabled:', diag.get('answer_synthesis_enabled'))
print('query rescue applied:', diag.get('query_rescue_applied'))
activity = diag.get('activity') or []
for step in activity:
    if isinstance(step, dict):
        print(' -', step.get('type'), '|', step.get('searchIndexArguments', {}).get('search') or step.get('knowledgeSourceName') or '', '| count=', step.get('count'))

mode: search_answer_synthesis
answer synthesis enabled: True
query rescue applied: None
 - modelQueryPlanning |  | count= None
 - searchIndex | high groundwater clay layers excavation risks design considerations | count= 50
 - searchIndex | excavation in high groundwater and clay layers key risks design considerations | count= 50
 - searchIndex | site with high groundwater and clay layers excavation support dewatering stability | count= 49
 - modelAnswerSynthesis |  | count= None
 - agenticReasoning |  | count= None


## Step 5 — Contrast with a single-shot mode

Run the same question with hybrid to feel the difference between fused single-query retrieval and planned multi-query retrieval.

In [5]:
resp_hybrid = lab.ask(Q2, job=job, retrieval_mode='hybrid', record_as='lab07_hybrid_q2')
lab.show_answer(resp_hybrid, max_citations=4)

[retrieval_mode=hybrid | scoring_profile=default | citations=8]

GEO Publication No. 1/2023, **Deep Excavation Design and Construction**, is a Hong Kong Geotechnical Engineering Office publication first published in December 2023. Loss of overall stability is likely to occur in excavations near a steeply-sloping site with a high groundwater table, or where a weak subsoil layer (e.g. loose sand or soft clay) is. [6]

Supporting evidence:
- In order to minimise disturbance to the adjacent ground due to the boring operation, the pressure of the compressed air should be carefully assessed and monitored, especially for sites with a high groundwater table and thick layers of loose fill, bouldery colluvium or rockfill. Trial boring is usually conducted and should be aimed to determine site-specific  [3]
- As discussed in Section 2.1.2, the differential piezometric pressures between marine clay and alluvial sand layers could lead to porewater pressure dissipation and associated consolidation s

## Takeaways

- Agentic retrieval **plans subqueries** and synthesizes a grounded, cited answer across sections.
- It typically returns more, better-targeted citations on multi-hop questions than single-shot hybrid.
- It is also the mode that can reach enrichment-index-only fields (e.g. OCR text from Lab 06).

Next: **Lab 08** swaps in the Content Understanding extractor and compares.